In [ ]:
import numpy as np
from numpy.random import default_rng

import matplotlib.pyplot as plt
from util.DataGen import nai_pulse
from util.Processing import discretize, td_nnlsr_deconvolve
from util.Plotting import plot_photons


In [ ]:
n = 80
bits = 8
gap = 5
mag = 20
ref_mag = 1000
spaces = np.arange(1,n,4)

_, kernel = nai_pulse(1)
trace = np.zeros((n+10) * kernel.size) #rouch in-exact preallocation to trim latter
volts_list = []
index_list = []

# Build directly so we can get good spacing between clusters for independence
start = 0
for spacing in spaces:
    # mag * kernel
    trace[start: start+kernel.size] += mag * kernel
    volts_list.append(mag)
    index_list.append(start)
    start+=spacing
    
    trace[start: start+kernel.size] += mag * kernel
    volts_list.append(mag)
    index_list.append(start)
    start+=kernel.size + gap
    
if start < trace.size: # trim a bit
    trace = trace[:start]
    
trace = discretize(trace, bits=bits)
energy_list = np.array(volts_list)
index_list = np.array(index_list)

print(energy_list)
print(index_list)

    
fig, axes = plt.subplots(3, 1, figsize=(8, 6), dpi=200)
# plot_photons(structure=axes[0], photons_=index_list, magnitude=energy_list, color='red',label_='Photons', alpha=1)
z = np.zeros_like(trace)
z[index_list] = mag
axes[0].plot(np.arange(trace.size), z, 'r.', label='True')
axes[0].legend()
axes[0].set_xlim(-10, trace.size)
axes[1].plot(trace, label='Trace')
axes[1].legend()
axes[1].set_xlim(-10, trace.size)


discretized_kernel = discretize(ref_mag*nai_pulse(1)[1], bits=bits)/ref_mag
deconv = td_nnlsr_deconvolve(trace, discretized_kernel)

gt0 = deconv > 0
ngt0 = deconv <= 0
axes[2].plot(np.arange(deconv.size)[gt0], deconv[gt0], marker='.', linestyle='', color='cyan', label='Deconvolution > 0')
axes[2].plot(np.arange(deconv.size)[ngt0], deconv[ngt0], marker='.', linestyle='', color='cyan', alpha=.01, label='Deconvolution = 0')
axes[2].set_xlim(-10, trace.size)
axes[2].legend()


In [ ]:
# TODO maybe we sweep 0-75 spacing with downsampling of 10. Split into multiple rows...